<a href="https://colab.research.google.com/github/vepharix/PythonProject/blob/main/Practical_Topics_01_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### MD6117 Machine Learning for Healthcare AI Practical: Topics 01 to 03

**Dataset:** Diabetes 130-US Hospitals, 1999 to 2008.

* 100,000 real inpatient encounters for patients with diabetes, drawn from 130 hospitals, with demographics, admission details, laboratory and medication counts, and the length of stay in days.

* Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008), mirrored on [Kaggle](https://www.kaggle.com/datasets/brandao/diabetes).

**Predict:** `time_in_hospital` - length of stay in days.

<details>
<summary><b>How to use this notebook (click to expand)</b></summary>

- Every heading starting with `#` or `##` can be **collapsed**. In Google Colab, click the small triangle to the left of a heading to fold that whole section away. Use this to hide parts you have finished so the notebook stays short.
- Sections in a grey box like this one are collapsible too. Click the summary line to open or close them.
- The notebook is organised as **Parts** (major sections) and **Tasks** inside them.
- Any code cell you are asked to complete contains `???` where something is missing.

</details>

## Set up

Run the cell below before anything else.

In [ ]:
# Run this cell FIRST. It installs the few libraries that are not already in Colab.
# It takes under a minute. You only need to run it once per session.

%pip install -q kagglehub ucimlrepo seaborn
print("Libraries ready. pandas, numpy, matplotlib and statsmodels are already installed.")

---
# Part A. Frame the clinical problem (Topic 01)

* A hospital group wants to anticipate how long diabetic inpatients will stay, so that bed managers can plan capacity and discharge teams can prioritise the patients who will need the most coordination.

* Before examining any data, a project must answer a few key questions. Getting these wrong is the most common reason healthcare AI projects fail, and no amount of modelling later can address a badly framed problem.

### TASK A1: Understand the problem

Fill in your team's answer for each blank in the cell below. Short phrases are enough.

* What exactly are we predicting: ???

* For which population: ???

* Who uses the output: ???

* Why is this important: ???

* Regression or classification: ???


### TEAM DISCUSSION A (5 minutes)

This dataset contains structured data: numbers and categories in a tidy table. However, real hospital data is typically unstructured.

1. Name three kinds of clinically important information about these admissions that this table did not contain.

---
# Part B. Obtain and explore the data (Topics 01 and 02)

Exploratory data analysis is the equivalent of taking a history and examining a patient before prescribing.

In [ ]:
## Install libraries
import os
import numpy as np
import pandas as pd ## Loading data and data wrangling
import matplotlib.pyplot as plt ## Plotting

In [ ]:
# WHAT THIS CELL DOES
#   Downloads the dataset and reads it into a table (a pandas DataFrame).
#   Two routes are provided because teaching venues differ: the Kaggle mirror first, and the official UCI repository as a fallback. You do not need to change anything.

try:
    import kagglehub
    path = kagglehub.dataset_download("brandao/diabetes")
    csv = [os.path.join(r, f) for r, _, fs in os.walk(path)
            for f in fs if f.endswith("diabetic_data.csv")]
    dataset_raw = pd.read_csv(csv[0])
    print("Loaded from the Kaggle mirror.")

except Exception:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=296)
    dataset_raw = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
    print("Loaded from the UCI repository.")

print("Rows and columns:", dataset_raw.shape)

In [ ]:
# WHAT THIS CELL DOES
#   A dataset can contain far more columns than any single analysis needs.
#   Selecting a subset to work on allows us to focus on the key data, provided that the selection is well-justified.
#   We keep an identifier and important information required to address the problem.


col_wanted = ["patient_nbr", "race", "gender", "age", "weight", "admission_type_id",
              "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
              "number_diagnoses", "number_outpatient", "number_inpatient", "number_emergency",
              "medical_specialty", "A1Cresult", "change", "diabetesMed", "readmitted"]

cols = [col for col in col_wanted if col in dataset_raw.columns]

df = dataset_raw[cols].copy()
print("Working table:", df.shape)
df.head()

---
# Part C. Data cleaning and preparation (Topic 02)

Exploratory data analysis is the equivalent of taking a history and examining a patient before prescribing.

* You are looking for missing data and anything implausible.

* Find all the problems before modelling anything.

In [ ]:
# WHAT THIS CELL DOES
#   .info() lists every column, its type, and how many non-null values it holds.
#   .describe() summarises the numeric columns.
#   Read both before forming any opinion about the data.

df.info()
df.describe().round(1)

### TASK C1: Checking for missing values

* `.info()` above suggests almost nothing is missing, but that is false.

* This dataset stores unknown values as the character `?`, but pandas counts it as a string.

* Replace the `???` to count how many `?` values sit in each column.

In [ ]:
# WHAT THIS CELL DOES
#   (df == "?") builds a table of True and False, and .sum() counts the Trues per column.
#   We keep only the columns where at least one missing placeholder was found.

missing_counts = (df == ???).sum()
print(missing_counts[missing_counts > 0])

print("\nAs a percentage of rows:")
print((missing_counts[missing_counts > 0] / len(df) * 100).round(1))

### TASK C2: Cleaning data

* Replace every `?` with a proper missing value marker so pandas can 'see' it properly.

* Fill in `???` with the proper missing value marker (numpy calls it `np.nan`).

In [ ]:
# .replace() swaps the placeholder for a real missing marker.
dataset_clean = df.replace("?", ???)

# Compute fraction missing per column
missing_fraction = dataset_clean.isna().mean()
missing_fraction

### TASK C3: Checking for incomplete records

* Before deleting anything, check whether missingness is spread evenly.

* Compare the average length of stay for encounters where `race` was recorded against those where it was not.

* Hint: `dataset_clean["race"].isna()` returns True where the value is missing.

In [ ]:
# WHAT THIS CELL DOES
#   .loc[condition, "column"] selects the rows meeting a condition and one column.
#   Comparing the outcome between the two groups tells you whether the gaps are random.

recorded = dataset_clean.loc[dataset_clean["race"].notna(), "time_in_hospital"].mean()
not_recorded = dataset_clean.loc[dataset_clean["race"].???(), "time_in_hospital"].mean()
n_missing = dataset_clean["race"].isna().sum()

print(f"Encounters with race recorded    : mean stay {recorded:.2f} days")
print(f"Encounters with race not recorded: mean stay {not_recorded:.2f} days ({n_missing} rows)")

### TASK C4: Checking for duplicates

* A patient can be admitted several times.

* Count the rows, count the distinct patients, and find the largest number of admissions for a single patient.

* Fill the `???` with the method that counts distinct values.

In [ ]:
# WHAT THIS CELL DOES
#   .nunique() counts how many distinct values a column holds.
#   .value_counts() counts how often each value appears, so its maximum is the number of encounters belonging to the most frequently admitted patient.

print("Rows (encounters)      :", len(dataset_clean))
print("Distinct patients      :", dataset_clean["patient_nbr"].???)
print("Most admissions by one patient:", dataset_clean["patient_nbr"].value_counts().???)

### TASK C5: Checking for outliers

* Look at the distinct values of `gender`.

* What is it, how many rows does it affect, and what would you do?

### TASK C6: Visualising the data (Topic 02) (10 minutes)

* A table of numbers tells you what is in the data. A plot tells you what shape it is in, and shape is what decides which model is appropriate.

* Each plot below answers one kind of question. Pick the plot by the question you are asking, not by which one looks best.

* Fill in the `???`. Use the cheatsheet for the code.

* Observe the plots and highlight any interesting insights.

| Question | Plot |
| --- | --- |
| What shape does one number have? | `???` |
| How spread out is it, and are there extreme values? | `???` |
| How many patients are in each category? | `???` |
| Do two numbers move together? | `???` |
| Does the outcome differ between groups? | `???` |
| Which numbers move together, all at once? | `???` |


In [ ]:
# WHAT THIS CELL DOES
#   Draws the six plots in the table above, on this dataset, in one figure.
#   Nothing to complete here. Read each panel, and note what it does and does not tell you.
#   These are the same one-line calls that appear on your cheatsheet.

fig, ax = plt.subplots(2, 3, figsize=(16, 8))

# 1. HISTOGRAM: what shape does one number have?
dataset_clean[???].plot(kind="hist", bins=14, ax=ax[0, 0])
ax[0, 0].set_title(???)
ax[0, 0].set_xlabel(???)

# 2. BOXPLOT: how spread out is it, and where are the extreme values?
dataset_clean.???(column=???, ax=ax[0, 1])
ax[0, 1].set_title(???)

# 3. BAR CHART: how many patients are in each category?
dataset_clean[???].value_counts().sort_index().plot(kind=???, ax=ax[0, 2])
ax[0, 2].set_title(???)
ax[0, 2].tick_params(axis="x", labelrotation=45)

# 4. SCATTER: do two numbers move together?
#    s (size) makes the dots small and alpha makes them see-through, because 100,000 solid dots would print as one black rectangle.
dataset_clean.plot(kind=???, x=???, y=???,
                   s=3, alpha=0.05, ax=ax[1, 0])
ax[1, 0].set_title(???)

# 5. GROUPED BOXPLOT: does the outcome differ between groups?
dataset_clean.boxplot(column=???, by=???, ax=ax[1, 1])
ax[1, 1].set_title(???)
ax[1, 1].set_xlabel(???)

# 6. CORRELATION HEATMAP: which numbers move together, all at once?
#    patient_nbr is an identifier, not a quantity, so it is removed first.
numeric = dataset_clean.select_dtypes(???).drop(columns=[???],
                                                     errors="ignore")
image = ax[1, 2].imshow(numeric.corr(), cmap="coolwarm", vmin=-1, vmax=1) ## Correlation
ax[1, 2].set_xticks(range(len(numeric.columns)))
ax[1, 2].set_xticklabels(numeric.columns, rotation=90, fontsize=7)
ax[1, 2].set_yticks(range(len(numeric.columns)))
ax[1, 2].set_yticklabels(numeric.columns, fontsize=7)
fig.colorbar(image, ax=ax[1, 2])
ax[1, 2].set_title(???)

plt.suptitle("")          # pandas adds its own title to a grouped boxplot
fig.tight_layout()
plt.show()

### TASK C7: The shape of what you are predicting

* A histogram sorts the values into bins and counts how many land in each one.

* Replace the `???` to draw the distribution of `time_in_hospital`.

* Then answer, in the cell below: is the average length of stay a fair summary of a typical patient?

In [ ]:
# WHAT THIS CELL DOES
#   Draws one numeric column as a histogram, then prints two summaries of the same
#   column so you can compare the picture with the numbers.

dataset_clean["time_in_hospital"].plot(kind=???, bins=14)
plt.xlabel("Length of stay (days)")
plt.ylabel("Number of encounters")
plt.show()

print("Mean  :", round(dataset_clean["time_in_hospital"].mean(), 2))
print("Median:", dataset_clean["time_in_hospital"].median())

### TASK C8: Spread and extreme values

* A boxplot shows the middle half of the data as a box, the median as a line inside it, and anything unusually far out as separate points.

* Replace the `???` to draw three numeric columns side-by-side.

* Then decide: are the points beyond the whiskers errors to remove?

In [ ]:
# WHAT THIS CELL DOES
#   Draws several numeric columns as boxplots on the same axis, so their spreads
#   can be compared directly.

cols_to_plot = ["num_medications", "num_lab_procedures", "number_diagnoses"]
dataset_clean[cols_to_plot].plot(kind=???)
plt.ylabel("Count per encounter")
plt.show()

### TASK C9: Does the outcome differ between groups?

* **No code is given for this task.** Use the *Visualisation* section of your coding cheatsheet.

* Draw one plot that answers: do encounters where diabetes medication was prescribed have a different length of stay from those where it was not?

* The columns you need are `time_in_hospital` and `diabetesMed`.

* Support the plot with one line that prints the group averages.

In [ ]:
# Write your own code here.
# The cheatsheet section you want is "Visualisation".
# You need one plotting line and one groupby line.

???

### TASK C10 [EXTENSION]: Do two numbers move together?

* **No code is given for this task.** Use the cheatsheet again.

* Draw a plot showing `num_medications` against `time_in_hospital`, then print the correlation between them.

* With around 100,000 rows, a plain scatter plot prints as a solid block. The cheatsheet shows the two arguments that fix it.

In [ ]:
# Write your own code here.
# One scatter plot, and one correlation.
# Look for s= and alpha= on the cheatsheet, and for .corr().

???

### TASK C11: Which columns move together

* Build the correlation heatmap for every numeric column, using the cheatsheet.

* Remember to remove `patient_nbr` first, and say why in a comment.

* Then find the pair of predictors with the strongest correlation, and say what trouble that pair could cause in the regression you are about to fit.

In [ ]:
# Write your own code here.
# Cheatsheet sections: "Correlation" and "Visualisation".

???

### TASK C12 [EXTENSION]: The same plots with less typing, using seaborn

* Everything so far used pandas and matplotlib, which is what the plot is made of.

* `seaborn` sits on top of matplotlib. You hand it the table and the column names, and it does the grouping, the summarising and the confidence intervals for you.

* Two things it gives you that are needed by hand: `hue=` splits any plot by a category, and several plot types show an uncertainty band without being asked.

* Replace the `???` to draw the annotated correlation heatmap, then compare each panel with the version you built by hand.

In [ ]:
# WHAT THIS CELL DOES
#   Redraws the plots from TASK C6 to C10 using seaborn, so you can see the difference in effort. Each panel is one line of plotting.

import seaborn as sns
sns.set_theme(style="whitegrid")

# seaborn plots every row it is given, so take a sample for the heavy panels.
sample = dataset_clean.sample(4000, random_state=42)

fig, ax = plt.subplots(2, 3, figsize=(17, 9))

# 1. Distribution, with a smoothed density curve on top
sns.histplot(data=dataset_clean, x="time_in_hospital", bins=14, kde=True, ax=ax[0, 0])
ax[0, 0].set_title("histplot: shape, with a density curve")

# 2. Grouped boxplot. One line, no suptitle to clean up afterwards.
sns.boxplot(data=dataset_clean, x="diabetesMed", y="time_in_hospital", ax=ax[0, 1])
ax[0, 1].set_title("boxplot: stay by diabetes medication")

# 3. Counts per category, ordered
sns.countplot(data=dataset_clean, x="age", ax=ax[0, 2],
              order=sorted(dataset_clean["age"].unique()))
ax[0, 2].set_title("countplot: encounters per age band")
ax[0, 2].tick_params(axis="x", labelrotation=45)

# 4. Scatter split by a category. hue= is the argument that earns seaborn its place.
sns.scatterplot(data=sample, x="num_medications", y="time_in_hospital",
                hue="diabetesMed", s=12, alpha=0.5, ax=ax[1, 0])
ax[1, 0].set_title("scatterplot with hue: split by a category")

# 5. A fitted line with a confidence band, computed for you
sns.regplot(data=sample, x="num_medications", y="time_in_hospital",
            scatter_kws={"s": 4, "alpha": 0.1}, line_kws={"color": "red"}, ax=ax[1, 1])
ax[1, 1].set_title("regplot: trend with a confidence band")

# 6. Correlation heatmap, with the numbers written in.
#    Replace the ??? with the seaborn function for a heatmap.
numeric = dataset_clean.select_dtypes("number").drop(columns=["patient_nbr"],
                                                     errors="ignore")
sns.???(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm",
        vmin=-1, vmax=1, annot_kws={"size": 6}, ax=ax[1, 2])
ax[1, 2].set_title("heatmap: correlation, with values")

plt.tight_layout()
plt.show()

### TEAM DISCUSSION C (10 minutes)

* How will your team tackle the missing values?

* Implement the code to tackle the missing values.

In [ ]:
dataset_clean = dataset_clean.???(columns=[???])
dataset_clean

---
# Part D. Train-Test Split (Topic 02)

The order of these steps decides whether your model score can be trusted.

* The rule: **split first, then fit every preparation step on the training data only.**

* Anything learned from the test set and used in training is leakage, and leakage inflates the model score (performance) in a way that will fail when deploying the model on real patients.

### TASK D1: Train-Test-Split by patient, not by row

* Divide the DISTINCT patients into 80 percent training and 20 percent test, then keep every encounter belonging to each group.

* Fill in the `???` with the column that identifies a patient.

In [ ]:
# WHAT THIS CELL DOES
#   We list the distinct patients, shuffle them, and take the first 80 percent for training.
#   .isin() then selects every ENCOUNTER whose patient is in that list.
#   This guarantees that no patient appears on both sides of the split.

from sklearn.model_selection import GroupShuffleSplit

# 1. Initialize 80/20 group split
train_size = 0.8
random_state = 2026
col_idx = "???"

dataset_clean_gss = GroupShuffleSplit(n_splits=1,
                                      train_size=train_size,
                                      random_state=random_state)

# 2. Split indices based on patient ID
train_idx, test_idx = next(dataset_clean_gss.split(dataset_clean, groups=dataset_clean[col_idx]))

# 3. Subset the dataframe to create the train set and test set
train = dataset_clean.iloc[train_idx]
test  = dataset_clean.iloc[test_idx]

# Sanity check: Verification
train_ids = set(train["patient_nbr"])
test_ids  = set(test["patient_nbr"])

print(f"{len(train)} training encounters from {len(train_ids)} patients")
print(f"{len(test)} test encounters from {len(test_ids)} patients")
print("Patients appearing in both:", len(train_ids & test_ids))

display(train)

### TASK D2: Learning only from training data

* Impute the remaining missing categorical values with the most common value in the **TRAINING** data, and apply that same value to the test data.

* Fill the `???` so the fill value is computed from the training set.

In [ ]:
# WHAT THIS CELL DOES
#   .mode()[0] is the most frequently occurring value in a column.
#   We learn it from the training data ONLY, then apply the same value to both sets.
#   This is because at prediction time, you cannot look at the test patients to decide anything.

train = train.copy(); test = test.copy()

for col in ["race"]:
    fill_value = ???[col].mode()[0]
    train[col] = train[col].fillna(fill_value)
    test[col]  = test[col].fillna(fill_value)
    print(f"{col}: filled with '{fill_value}'")

print("Remaining missing values, training:", int(train.isna().sum().sum()))

### TEAM DISCUSSION D (3 minutes)

* Suppose you had imputed with a fill value or use a scaling factor, from the whole table before splitting. Your model score would probably look better.

* Why is that number not trustworthy, and which set should the value have been learned from?

---
# Part E. Training a regression model (Topic 03)

* A linear regression is like a sentence with numbers in it.

* We use `statsmodels` because it prints the full coefficient table, with uncertainty and p values, in the format used in clinical papers.

In [ ]:
# WHAT THIS CELL DOES
#   statsmodels lets you write the model as a formula: outcome (y column) ~ predictors (X columns).
#   C(...) marks a column as categorical, so it is turned into indicator variables automatically and one level is used as the reference.
#   .fit() trains the model and estimates the coefficients from the training data only.

import statsmodels.formula.api as smf

formula = ("time_in_hospital ~ num_medications + num_lab_procedures + num_procedures "
           "+ number_diagnoses + number_inpatient + C(diabetesMed)")

model = smf.ols(formula, data=train).fit()

print(model.summary().tables[1])
print("R-squared:", round(model.rsquared, 3))

### TASK E1: Explain the model

* Complete the two sentences using the numbers in the table above. Keep the units: these are days of stay.

In [ ]:
print("Each additional medication is associated with about ??? more days in hospital.")
print("Each additional diagnosis recorded is associated with about ??? more days in hospital.")

### TASK E2: Is the medication effect causal?

* Answer in one sentence: if a doctor prescribed fewer medications tomorrow, would that patient go home sooner?

In [ ]:
answer = "???"
print(answer)

### TASK E3 [EXTENSION]: Evaluting goodness of fit
* Compare candidate models by balancing goodness of fit against complexity (number of estimated parameters). Lower values are better.
* AIC (Akaike Information Criterion): Penalise extra complexity moderately; often used when prediction is the main goal.
    * AIC = −2 log(_L_) + 2*k*
    * _L_ is likelihood, _k_ is the number of parameters, and _n_ is the sample size
* BIC (Bayesian Information Criterion): Penalise extra complexity more strongly, especially with larger samples; it tends to prefer simpler models.
    * BIC = −2 log(_L_)+ _k_ log(_n_)


* Compare models fitted to the same outcome and same observations.
* Select the model with the lowest AIC or lowest BIC.
* Do not interpret a single AIC or BIC value as "good" or "bad" on its own.
* If AIC and BIC disagree, AIC supports a more complex predictive model, while BIC supports a simpler model.

In [ ]:
print("AIC:", model.aic)
print("BIC:", model.bic)

### TASK E4 [EXTENSION]: A predictor with a large `p` value

* Try different equations and find a term in the summary table whose `p` value is above 0.05.
* Should you delete that variable? Write your team's answer and say what you would check first.

---
# Part F. Evaluate the model (Topic 03)

* A model is judged on patients it has never seen, using an error measure a bed manager can act on.

### TASK F1: Evaluation metric on the held-out test set

* Compute the error (loss) between the predicted values (`y_pred`) and the actual groundtruth values (`y`), using MAE, MSE, and RMSE on the test encounters.

* Fill the `???`.

In [ ]:
# WHAT THIS CELL DOES
#   .predict() applies the fitted model to patients it has never seen.
#   The error is actual minus predicted.
#   MAE averages the size of the misses;
#   MSE squares the size of the misses;
#   RMSE squares them first, so a few large misses count for more, before taking a square root.

## Obtaining the prediction of y_pred (pred) using X_test (test)
y_pred = model.predict(test)

## Calculating the error by hand so that we understand how it is calculated
error = test["time_in_hospital"] - y_pred

MAE  = ???
print(f"MAE : {MAE:.2f} days")

MSE = ???
print(f"MSE: {MSE:.2f} days")

RMSE = ???
print(f"RMSE: {RMSE:.2f} days")

print(f"For comparison, the average stay is {test['time_in_hospital'].mean():.2f} days")

In [ ]:
## Using evaluation metric from sk-learn library
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

col_y = "time_in_hospital"
y_test = test[col_y] ## Obtaining the groundtruth column y_test

## Obtaining the prediction (y) for X_test
y_pred = model.predict(test)

MAE = mean_absolute_error(y_test, y_pred)
print(f"MAE : {MAE:.2f} days")

MSE = mean_squared_error(y_test, y_pred)
print(f"MSE: {MSE:.2f} days")

RMSE = root_mean_squared_error(y_test, y_pred)
print(f"RMSE: {RMSE:.2f} days")

### TASK F2: Look at the errors, not just the score

* Plot the residuals against the fitted values, then describe the shape in one word.

In [ ]:
# WHAT THIS CELL DOES
#   A residual is actual minus predicted, so it is what the model got wrong.
#   Plotting residuals against predictions shows whether the errors are random (good) or follow a pattern (something systematic is missing from the model).
y_train = model.predict(train)
y_train_resid = train[col_y] - y_train
plt.figure(figsize=(8, 4))
plt.scatter(y_train, y_train_resid, s=4, alpha=0.2)
plt.axhline(0, color="red")
plt.xlabel("Predicted length of stay (days)"); plt.ylabel("Actual minus predicted")

### TASK F3: The four assumptions behind a linear regression

* Ordinary least squares rests on four assumptions, remembered as **LINE**:

| | Assumption | How you check it |
| --- | --- | --- |
| **L** | The mean relationship is **linear** | Residuals vs fitted values show no systematic curve; they should be centred around 0. |
| **I** | The errors are **independent** of each other | No autocorrelation or clustering (e.g., repeated subjects) |
| **N** | The errors are **normally** distributed | Q-Q points approximately follow the reference line |
| **E** | The errors have **equal variance** (homoscedasticity) | Residual spread remains roughly constant across fitted values; spread of residuals does not widen |

* Only two of these change the coefficients. The other two change how much you can trust the p values and confidence intervals, which is a different kind of damage.

* Replace the `???` to draw the Q-Q plot, then read all four panels.

In [ ]:
# WHAT THIS CELL DOES
#   Draws one panel for each of the four assumptions.
#   A Q-Q plot sorts the residuals and plots them against the values you would expect if they really were normal.
#   If they are, the points sit on the diagonal line.

import statsmodels.api as sm

resid  = model.resid
fitted = model.fittedvalues

fig, ax = plt.subplots(2, 2, figsize=(12, 8))

# L: Residuals vs fitted values
ax[0, 0].scatter(fitted, resid, s=3, alpha=0.1)
ax[0, 0].axhline(0, color="red", lw=1)
ax[0, 0].set_title("L: Linearity")
ax[0, 0].set_xlabel("Fitted length of stay (predicted days)")
ax[0, 0].set_ylabel("Residual (observed − predicted days)")
# Each dot is an encounter.
# The x-axis is predicted length of stay; the y-axis is the error in days: observed stay minus predicted stay.
# We want a patternless horizontal cloud centred on zero; a curve would indicate that the linear model misses structure.
# Residual-versus-fitted plots are used to identify non-linearity.

# I: Residuals in dataset row order
ax[0, 1].plot(resid.values[:300], lw=0.8)
ax[0, 1].axhline(0, color="red", lw=1)
ax[0, 1].set_title("I: Independence — only checks row-order patterns")
ax[0, 1].set_xlabel("Encounter row number (first 300 rows, current dataset order)")
ax[0, 1].set_ylabel("Residual (observed − predicted days)")
# Each point is an encounter residual plotted in the dataset’s current row order.
# Look for runs: many consecutive residuals on the same side of zero—for example, a long stretch of positive residuals followed by a long stretch of negative residuals.
# This suggests that nearby rows have more similar errors than expected by chance.
#
# Look for drift: residuals that gradually move upward or downward over the row sequence, rather than repeatedly returning to zero.
# A drift may indicate a time trend, workflow change, or other ordering effect that the model has not captured.
#
# Look for waves: repeated rises and falls across the sequence. This may indicate cyclic or time-related dependence.
#
# However, this plot does not test the clinically important dependence here: multiple encounters from the same patient.
# Residuals from one patient can be correlated even when the line appears random, particularly if rows are not ordered by patient and time.
# Repeated measures and patient-level clustering are therefore sources of non-independent errors.


# N: Normal Q-Q plot
sm.???(resid, line="45", fit=True, ax=ax[1, 0])
ax[1, 0].set_title("N: Normality of residuals")
ax[1, 0].set_xlabel("Theoretical normal quantiles")
ax[1, 0].set_ylabel("Observed residual quantiles")
# Each point compares an ordered observed residual with the matching normal-distribution quantile.
# The x-axis gives quantiles expected from a normal distribution; the y-axis gives the matching ordered residual quantiles from the model.
# Points near the diagonal indicate approximately normal residuals; an upward departure in the upper tail indicates unusually large positive residuals.

# E: Scale-location plot
std_resid = resid / resid.std()
ax[1, 1].scatter(fitted, np.sqrt(np.abs(std_resid)), s=3, alpha=0.1)
ax[1, 1].set_title("E: Equal variance (scale–location plot)")
ax[1, 1].set_xlabel("Fitted length of stay (predicted days)")
ax[1, 1].set_ylabel("Square root of |standardized residual|")
# Each dot is an encounter.
# The x-axis is predicted length of stay, while the y-axis is the size—not direction—of its standardized residual.
# A level band supports constant error variance; a rising trend means prediction errors become more variable for longer predicted stays.

plt.tight_layout()
plt.show()



### TASK F4 [EXTENSION]: Apply transformation

* Refit the model on the logarithm of length of stay and compare the two residual plots.
* Do the points spread out evenly?
* What might be a challenge in interpreting this new model?


### TEAM DISCUSSION F (5 minutes)

Your team must hand a hospital bed manager exactly one number.

1. Which number: MAE or MSE or RMSE, and why that one for bed planning?
2. Examine the R^2 value and explain how to interpret it. Is this model useful for planning? Give a reason.
3. Examine the error. Is this model useful for planning? Give a reason.

---
# Wrap-up

* In this session, you have examined a clinical question and worked out the whole machine learning pipeline:
framed the problem and its user,
explored a real hospital extract,
cleaned the data,
split by patient,
fitted a regression model,
explained the model,
and tested it on patients it had never seen.

* This sequence forms the backbone of most machine learning projects. There are various approaches to get a better model, and this process is iterative.